# Problem 1: Multi-agent TLC Eluent Optimization App

In this homework, we build a **multi-agent application** that mimics how a chemist reasons about TLC optimization.

The application:
1. Accepts a reaction SMILES (e.g., `CC(=O)O.CCO>>CC(=O)OCC`)
2. Identifies reactants and products
3. Uses three cooperating agents to iteratively search for the optimal TLC eluent composition

We use the TLC neural network model trained in the previous homework (Class 3-5) as a **TLC emulator** to simulate experimental Rf values.

In [1]:
import json
import yaml
import numpy as np
from openai import OpenAI
from TLC_pred import get_model, predict_rf

## Setup: Load the TLC Emulator

We load the pre-trained TLC neural network model from `TLC_pred.py`. If no saved model exists on disk, it will be trained automatically and saved for future use.

In [2]:
nn_model = get_model()
print("TLC emulator is ready (loaded cached model or trained automatically).")

Training rows used: 4944
Epoch   1/200 | MSE Loss: 0.1495
Epoch  10/200 | MSE Loss: 0.0848
Epoch  20/200 | MSE Loss: 0.0655
Epoch  30/200 | MSE Loss: 0.0544
Epoch  40/200 | MSE Loss: 0.0465
Epoch  50/200 | MSE Loss: 0.0410
Epoch  60/200 | MSE Loss: 0.0371
Epoch  70/200 | MSE Loss: 0.0343
Epoch  80/200 | MSE Loss: 0.0320
Epoch  90/200 | MSE Loss: 0.0306
Epoch 100/200 | MSE Loss: 0.0294
Epoch 110/200 | MSE Loss: 0.0286
Epoch 120/200 | MSE Loss: 0.0278
Epoch 130/200 | MSE Loss: 0.0272
Epoch 140/200 | MSE Loss: 0.0270
Epoch 150/200 | MSE Loss: 0.0264
Epoch 160/200 | MSE Loss: 0.0262
Epoch 170/200 | MSE Loss: 0.0257
Epoch 180/200 | MSE Loss: 0.0255
Epoch 190/200 | MSE Loss: 0.0252
Epoch 200/200 | MSE Loss: 0.0251
Finished training TLC model. Test MSE: 0.0240
Saved trained TLC model to: /Users/sunxueli/Desktop/ML Class 6 Homework/code/tlc_model.pkl
TLC emulator is ready (loaded cached model or trained automatically).


In [3]:
# Quick sanity check: Rf should increase as ethyl acetate fraction increases
test_smiles = 'COC1=CC(Cl)=CC=C1'
print(f"Rf vs. eluent polarity for {test_smiles}:")
for ea in [5, 10, 20, 30, 50, 80]:
    rf = predict_rf(nn_model, test_smiles, 100 - ea, ea)
    print(f"  Hexane {100-ea}% / EA {ea}% -> Rf = {rf:.4f}")

Rf vs. eluent polarity for COC1=CC(Cl)=CC=C1:
  Hexane 95% / EA 5% -> Rf = 0.3497
  Hexane 90% / EA 10% -> Rf = 0.3820
  Hexane 80% / EA 20% -> Rf = 0.4456
  Hexane 70% / EA 30% -> Rf = 0.5184
  Hexane 50% / EA 50% -> Rf = 0.6663
  Hexane 20% / EA 80% -> Rf = 0.8851


## Q1: Build Three Agents

We implement three agents using the OpenAI-compatible LLM API:

1. **Explanation Agent** - Interprets TLC results and decides if separation is acceptable
2. **Composition Tuning Agent** - Proposes the next eluent composition
3. **Experimentation Agent** - Assembles simulated TLC results into a structured report

In [4]:
# 用你自己的API与url
client = OpenAI(
    api_key="61caf8658ef54a5f928be8688a708f9f.SkQC2mOy97AYN76p",
    base_url="https://open.bigmodel.cn/api/paas/v4"
)

MODEL = "glm-5"

def parse_json_response(text):
    """Extract JSON from LLM response, handling markdown code blocks."""
    text = text.strip()
    if text.startswith('```'):
        lines = text.split('\n')
        end_idx = len(lines) - 1
        for i in range(len(lines) - 1, 0, -1):
            if lines[i].strip() == '```':
                end_idx = i
                break
        text = '\n'.join(lines[1:end_idx])
    return json.loads(text)


def call_agent(system_prompt, user_message, retries=2):
    """Call an LLM agent and return parsed JSON response."""
    for attempt in range(retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.3,
                response_format={"type": "json_object"}
            )
            return parse_json_response(response.choices[0].message.content)
        except Exception as e:
            if attempt < retries:
                print(f"  [Retry {attempt+1}] {e}")
            else:
                raise

In [5]:
with open('agent_prompts.yaml', 'r', encoding='utf-8') as f:
    agent_config = yaml.safe_load(f)

EXPLANATION_SYSTEM_PROMPT = agent_config['explanation_agent']['system_prompt']
EXPLANATION_INITIAL_PROMPT = agent_config['explanation_agent']['initial_prompt']
COMPOSITION_SYSTEM_PROMPT = agent_config['composition_agent']['system_prompt']
EXPERIMENTATION_SYSTEM_PROMPT = agent_config['experimentation_agent']['system_prompt']

print("Agent prompts loaded from agent_prompts.yaml")

Agent prompts loaded from agent_prompts.yaml


In [6]:
#  Agent testing: staged tests (local -> API -> mini pipeline)
from openai import AuthenticationError

def parse_reaction_smiles_local(reaction_smiles):
    """Parse reaction SMILES into reactants list and product."""
    parts = reaction_smiles.split('>>')
    reactants = parts[0].split('.')
    products = parts[1].split('.')
    return reactants, products[0]

def _print_title(name):
    print("\n" + "=" * 70)
    print(name)

def test_agents_once(reaction_smiles="CC(=O)O.CCO>>CC(=O)OCC"):
    reactants, product = parse_reaction_smiles_local(reaction_smiles)

    _print_title("Test 1: JSON parser (local, no API)")
    sample = """```json\n{\"ok\": true, \"value\": 1}\n```"""
    parsed = parse_json_response(sample)
    print("Parsed:", parsed)

    _print_title("Test 2: API connectivity (minimal)")
    try:
        ping = call_agent(
            "You are a strict JSON assistant.",
            "Return JSON with keys: ok (bool), msg (string)."
        )
        print("API connectivity OK:", ping)
    except AuthenticationError as e:
        print("API auth failed (401). 请更新 api_key 后重试。")
        print("Error:", e)
        return None
    except Exception as e:
        print("API connectivity failed:", e)
        return None

    _print_title("Test 3: Explanation Agent")
    expl_msg = EXPLANATION_INITIAL_PROMPT.format(
        reaction_smiles=reaction_smiles,
        product=product,
        reactants=', '.join(reactants)
    )
    explanation_result = call_agent(EXPLANATION_SYSTEM_PROMPT, expl_msg)
    print(json.dumps(explanation_result, indent=2, ensure_ascii=False))

    _print_title("Test 4: Composition Agent")
    comp_msg = (
        f"Product SMILES: {product}\\n"
        f"Reactant SMILES: {', '.join(reactants)}\\n\\n"
        f"Explanation Agent analysis:\\n{json.dumps(explanation_result, indent=2)}\\n\\n"
        f"History of all compositions tried so far:\\n[]"
    )
    composition_result = call_agent(COMPOSITION_SYSTEM_PROMPT, comp_msg)
    print(json.dumps(composition_result, indent=2, ensure_ascii=False))

    hexane = composition_result['proposed_eluent']['hexane']
    ea = composition_result['proposed_eluent']['ethyl_acetate']
    product_rf = predict_rf(nn_model, product, hexane, ea)
    reactant_rfs = {r: predict_rf(nn_model, r, hexane, ea) for r in reactants}

    _print_title("Test 5: Experimentation Agent")
    exp_msg = (
        f"Product SMILES: {product}\\n"
        f"Reactant SMILES: {', '.join(reactants)}\\n"
        f"Eluent tested: Hexane {hexane}% / Ethyl Acetate {ea}%\\n\\n"
        f"Simulated Rf values:\\n"
        f"  Product ({product}): {product_rf:.4f}\\n"
    )
    for r in reactants:
        exp_msg += f"  Reactant ({r}): {reactant_rfs[r]:.4f}\\n"
    experiment_result = call_agent(EXPERIMENTATION_SYSTEM_PROMPT, exp_msg)
    print(json.dumps(experiment_result, indent=2, ensure_ascii=False))

    _print_title("Test summary")
    print(f"Eluent: Hexane {hexane}% / EA {ea}%")
    print(f"Product Rf: {product_rf:.4f}")
    for r in reactants:
        print(f"Reactant {r}: Rf={reactant_rfs[r]:.4f}, delta={abs(product_rf-reactant_rfs[r]):.4f}")

    return {
        "explanation": explanation_result,
        "composition": composition_result,
        "experiment": experiment_result,
        "rf": {
            "product": product_rf,
            "reactants": reactant_rfs,
        },
    }

# Run one-shot staged tests
agent_test_result = test_agents_once()


Test 1: JSON parser (local, no API)
Parsed: {'ok': True, 'value': 1}

Test 2: API connectivity (minimal)
API connectivity OK: {'ok': True, 'msg': 'Success'}

Test 3: Explanation Agent
{
  "separation_assessment": "Task acknowledged: Separate Ethyl acetate (product) from Acetic acid and Ethanol (reactants). Ethyl acetate is an ester with moderate polarity, whereas Acetic acid and Ethanol are highly polar protic compounds that interact strongly with silica gel. Consequently, the product is expected to travel faster (higher Rf) than the reactants in normal-phase TLC. No experimental data is available yet.",
  "meets_product_rf_target": false,
  "meets_separation_target": false,
  "overall_status": "not_acceptable",
  "next_step_guidance": "Propose a reasonable starting eluent composition, such as 10-20% Ethyl Acetate in Hexane. Since the product is less polar than the reactants, it will migrate further; this composition range is a standard starting point to bring the product into the tar

## Q2: Connect the Agents into an Iterative Workflow

The optimization loop works as follows:

1. **User** sends a task request to separate product using TLC (provides a reaction SMILES)
2. **Explanation Agent** interprets the results (first iteration: receives the user task description to start)
3. **Composition Tuning Agent** proposes a better eluent composition
4. **TLC Emulator** (our trained model) predicts Rf values for all compounds
5. **Experimentation Agent** assembles the raw results into a structured report
6. Go back to step 2 and repeat until the stopping criterion is satisfied or the maximum iteration number is reached

In [7]:
def parse_reaction_smiles(reaction_smiles):
    """Parse reaction SMILES into reactants list and product."""
    parts = reaction_smiles.split('>>')
    reactants = parts[0].split('.')
    products = parts[1].split('.')
    return reactants, products[0]


def run_tlc_optimization(reaction_smiles, max_iterations=5, verbose=True):
    """Run the multi-agent TLC eluent optimization loop."""
    reactants, product = parse_reaction_smiles(reaction_smiles)

    if verbose:
        print(f"Reaction SMILES: {reaction_smiles}")
        print(f"Product: {product}")
        print(f"Reactants: {reactants}")

    history = []
    explanation_result = None
    experiment_result = None

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n{'=' * 70}")
            print(f"  Iteration {iteration + 1}")
            print(f"{'=' * 70}")

        # --- Step 1: Explanation Agent interprets results ---
        if iteration == 0:
            expl_msg = EXPLANATION_INITIAL_PROMPT.format(
                reaction_smiles=reaction_smiles,
                product=product,
                reactants=', '.join(reactants)
            )
        else:
            expl_msg = (
                f"Product SMILES: {product}\n"
                f"Reactant SMILES: {', '.join(reactants)}\n"
                f"Target: product Rf 0.20-0.30, each reactant differs from "
                f"product by >0.10 in Rf.\n\n"
                f"Experiment results:\n{json.dumps(experiment_result, indent=2)}\n\n"
                f"History of all iterations so far:\n{json.dumps(history, indent=2)}"
            )

        explanation_result = call_agent(EXPLANATION_SYSTEM_PROMPT, expl_msg)
        if verbose:
            print("\n[Explanation Agent]")
            print(f"  Assessment: {explanation_result.get('separation_assessment', '')}")
            print(f"  Product Rf OK? {explanation_result.get('meets_product_rf_target')}")
            print(f"  Separation OK? {explanation_result.get('meets_separation_target')}")
            print(f"  Status: {explanation_result.get('overall_status')}")
            print(f"  Guidance: {explanation_result.get('next_step_guidance', '')}")

        if iteration > 0 and explanation_result.get('overall_status') == 'acceptable':
            if verbose:
                prev = history[-1]
                print(f"\n>>> ACCEPTABLE separation found! "
                      f"Hexane {prev['hexane']}% / EA {prev['ethyl_acetate']}% <<<")

            # Update the last record in place so final summary status is consistent.
            history[-1]['status'] = 'acceptable'
            history[-1]['accepted_at_iteration'] = iteration + 1
            break

        # --- Step 2: Composition Tuning Agent proposes eluent ---
        comp_msg = (
            f"Product SMILES: {product}\n"
            f"Reactant SMILES: {', '.join(reactants)}\n\n"
            f"Explanation Agent analysis:\n{json.dumps(explanation_result, indent=2)}\n\n"
            f"History of all compositions tried so far:\n{json.dumps(history, indent=2)}"
        )

        composition_result = call_agent(COMPOSITION_SYSTEM_PROMPT, comp_msg)
        if verbose:
            print("\n[Composition Agent]")
            print(f"  Proposed: Hexane {composition_result['proposed_eluent']['hexane']}% "
                  f"/ EA {composition_result['proposed_eluent']['ethyl_acetate']}%")
            print(f"  Direction: {composition_result['change_direction']}")
            print(f"  Reason: {composition_result['reason']}")

        if composition_result.get('converged', False) and iteration > 0:
            if verbose:
                print("\n>>> CONVERGED! The current composition satisfies all criteria. <<<")
            break

        hexane = composition_result['proposed_eluent']['hexane']
        ea = composition_result['proposed_eluent']['ethyl_acetate']

        # --- Step 3: TLC Emulator predicts Rf values ---
        product_rf = predict_rf(nn_model, product, hexane, ea)
        reactant_rfs = {r: predict_rf(nn_model, r, hexane, ea) for r in reactants}
        delta_rfs = {r: abs(product_rf - rf) for r, rf in reactant_rfs.items()}

        if verbose:
            print("\n[TLC Emulator]")
            print(f"  Product Rf = {product_rf:.4f}")
            for r in reactants:
                print(f"  Reactant {r}: Rf = {reactant_rfs[r]:.4f}, "
                      f"delta_Rf = {delta_rfs[r]:.4f}")

        # --- Step 4: Experimentation Agent assembles report (no history) ---
        exp_msg = (
            f"Product SMILES: {product}\n"
            f"Reactant SMILES: {', '.join(reactants)}\n"
            f"Eluent tested: Hexane {hexane}% / Ethyl Acetate {ea}%\n\n"
            f"Simulated Rf values:\n"
            f"  Product ({product}): {product_rf:.4f}\n"
        )
        for r in reactants:
            exp_msg += f"  Reactant ({r}): {reactant_rfs[r]:.4f}\n"

        experiment_result = call_agent(EXPERIMENTATION_SYSTEM_PROMPT, exp_msg)
        if verbose:
            print("\n[Experimentation Agent]")
            print(f"  Summary: {experiment_result.get('experiment_summary', '')}")

        # Record history
        history.append({
            "iteration": iteration + 1,
            "hexane": hexane,
            "ethyl_acetate": ea,
            "product_rf": round(product_rf, 4),
            "reactant_rfs": {k: round(v, 4) for k, v in reactant_rfs.items()},
            "delta_rfs": {k: round(v, 4) for k, v in delta_rfs.items()},
            "status": explanation_result.get('overall_status', 'unknown')
        })

    return history

## Q3: Optimize the TLC Eluent

We test on two widely used reactions in drug synthesis. The goal is to find a hexane/ethyl acetate composition such that:
- Product Rf is between **0.20 and 0.30**
- All reactants have $$|\Delta R_f| > 0.10$$ compared to the product

In [ ]:
# Buchwald-Hartwig 交叉偶联案例测试
reactant_1 = "Cc1ccc(Cl)cc1"
product_2 = "Cc1ccc(N2CCOCC2)cc1"
reaction_smiles_bh = f"{reactant_1}>>{product_2}"

print("Testing reaction:", reaction_smiles_bh)
history_bh = []
try:
    history_bh = run_tlc_optimization(reaction_smiles_bh, max_iterations=5, verbose=True)
except Exception as e:
    print("\nOptimization stopped with error:", e)

print("\n" + "=" * 70)
print("Buchwald-Hartwig case summary")
print("=" * 70)
if len(history_bh) == 0:
    print("No valid iteration history was produced.")
else:
    display(history_bh)
    final_try = history_bh[-1]
    print(
        f"Final suggested eluent: Hexane {final_try['hexane']}% / "
        f"EA {final_try['ethyl_acetate']}%"
    )
    print(
        f"Final product Rf: {final_try['product_rf']:.4f}; "
        f"Status from previous explanation step: {final_try['status']}"
    )

# 便于后续单元复用
history = history_bh

Testing reaction: Cc1ccc(Cl)cc1>>Cc1ccc(N2CCOCC2)cc1
Reaction SMILES: Cc1ccc(Cl)cc1>>Cc1ccc(N2CCOCC2)cc1
Product: Cc1ccc(N2CCOCC2)cc1
Reactants: ['Cc1ccc(Cl)cc1']

  Iteration 1
